# HIT-UAV Thermal YOLO Fine-Tuning on Kaggle

This notebook fine-tunes a YOLO11 detector on HIT-UAV thermal drone imagery for the SAR footage demo.

Kaggle setup:
1. Create a Kaggle Notebook.
2. Turn on GPU: **Settings -> Accelerator -> GPU T4 x2** or any available GPU.
3. Attach your HIT-UAV dataset. The dataset should contain:

```text
images/train
images/val
images/test
labels/train
labels/val
labels/test
```

The notebook will create a Kaggle-safe dataset YAML in `/kaggle/working/` and save results there.

In [ ]:
# Kaggle already includes most ML/vision dependencies. Install Ultralytics without
# letting pip rewrite Kaggle's RAPIDS/CUDA stack, which can trigger dask-cuda/cudf warnings.
!pip install -q --no-deps ultralytics ultralytics-thop
!pip install -q --no-deps pyyaml

In [ ]:
from pathlib import Path
from collections import Counter
import random
import shutil
import yaml
import torch
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## Configuration

If auto-discovery fails, set `DATASET_ROOT` manually to the folder that contains `images/` and `labels/`.

In [ ]:
# Set to None for auto-discovery, or set manually, for example:
# DATASET_ROOT = Path('/kaggle/input/hit-uav')
DATASET_ROOT = None

PROJECT_DIR = Path('/kaggle/working/runs/detect')
RUN_NAME = 'hit_uav_yolo11s'
DATA_YAML = Path('/kaggle/working/hit_uav_thermal.yaml')

CLASS_NAMES = {
    0: 'Person',
    1: 'Car',
    2: 'Bicycle',
    3: 'OtherVehicle',
    4: 'DontCare',
}

# Training defaults. Lower IMG_SIZE/BATCH if Kaggle runs out of memory.
BASE_MODEL = 'yolo11s.pt'
IMG_SIZE = 960
EPOCHS = 60
BATCH = 8
WORKERS = 2
PATIENCE = 15

In [ ]:
def looks_like_yolo_root(path: Path) -> bool:
    return (
        (path / 'images' / 'train').exists()
        and (path / 'images' / 'val').exists()
        and (path / 'labels' / 'train').exists()
        and (path / 'labels' / 'val').exists()
    )


def discover_dataset_root() -> Path:
    candidates = []
    for root in Path('/kaggle/input').glob('*'):
        if looks_like_yolo_root(root):
            candidates.append(root)
        for child in root.rglob('*'):
            if child.is_dir() and looks_like_yolo_root(child):
                candidates.append(child)
    if not candidates:
        raise FileNotFoundError('Could not find a YOLO dataset root under /kaggle/input. Set DATASET_ROOT manually.')
    candidates = sorted(set(candidates), key=lambda p: len(str(p)))
    return candidates[0]


if DATASET_ROOT is None:
    DATASET_ROOT = discover_dataset_root()
else:
    DATASET_ROOT = Path(DATASET_ROOT)

print('Dataset root:', DATASET_ROOT)
assert looks_like_yolo_root(DATASET_ROOT), f'Not a YOLO root: {DATASET_ROOT}'

## Write Kaggle Dataset YAML

In [ ]:
data_yaml = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': CLASS_NAMES,
    'nc': len(CLASS_NAMES),
}

with DATA_YAML.open('w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(DATA_YAML.read_text())

## Dataset Sanity Check

In [ ]:
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def count_images(split: str) -> int:
    return sum(1 for p in (DATASET_ROOT / 'images' / split).rglob('*') if p.suffix.lower() in image_exts)

def count_labels(split: str) -> int:
    return sum(1 for p in (DATASET_ROOT / 'labels' / split).rglob('*.txt'))

def class_counts(split: str) -> Counter:
    counts = Counter()
    for label_file in (DATASET_ROOT / 'labels' / split).rglob('*.txt'):
        for line in label_file.read_text().splitlines():
            if not line.strip():
                continue
            counts[int(float(line.split()[0]))] += 1
    return counts

for split in ['train', 'val', 'test']:
    print(split)
    print('  images:', count_images(split))
    print('  labels:', count_labels(split))
    print('  class boxes:', {CLASS_NAMES[k]: v for k, v in sorted(class_counts(split).items())})

## Preview A Few Labels

In [ ]:
def yolo_to_xyxy(row, width, height):
    cls, xc, yc, bw, bh = row
    x1 = int((xc - bw / 2) * width)
    y1 = int((yc - bh / 2) * height)
    x2 = int((xc + bw / 2) * width)
    y2 = int((yc + bh / 2) * height)
    return int(cls), x1, y1, x2, y2


sample_images = list((DATASET_ROOT / 'images' / 'train').glob('*.jpg'))
sample_images = random.sample(sample_images, min(6, len(sample_images)))

plt.figure(figsize=(15, 8))
for idx, image_path in enumerate(sample_images, start=1):
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    label_path = DATASET_ROOT / 'labels' / 'train' / f'{image_path.stem}.txt'
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = [float(x) for x in line.split()]
            cls, x1, y1, x2, y2 = yolo_to_xyxy(parts, w, h)
            color = (0, 255, 255) if cls == 0 else (255, 180, 0)
            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(image, CLASS_NAMES[cls], (x1, max(15, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    plt.subplot(2, 3, idx)
    plt.imshow(image)
    plt.axis('off')
    plt.title(image_path.name)
plt.tight_layout()
plt.show()

## Train YOLO11s

For a quick smoke test, set `EPOCHS = 2`, `IMG_SIZE = 640`, and `BASE_MODEL = 'yolo11n.pt'` above. For the real run, use the defaults.

In [ ]:
model = YOLO(BASE_MODEL)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    workers=WORKERS,
    patience=PATIENCE,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    cache=False,
)

results

## Validate Best Weights

In [ ]:
best_weights = PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt'
last_weights = PROJECT_DIR / RUN_NAME / 'weights' / 'last.pt'

print('best:', best_weights, best_weights.exists())
print('last:', last_weights, last_weights.exists())

trained_model = YOLO(str(best_weights))
metrics_val = trained_model.val(data=str(DATA_YAML), split='val', imgsz=IMG_SIZE, device=0 if torch.cuda.is_available() else 'cpu')
metrics_test = trained_model.val(data=str(DATA_YAML), split='test', imgsz=IMG_SIZE, device=0 if torch.cuda.is_available() else 'cpu')

## Run Sample Predictions

In [ ]:
predict_dir = Path('/kaggle/working/predictions')
test_images = list((DATASET_ROOT / 'images' / 'test').glob('*.jpg'))
test_sample = random.sample(test_images, min(16, len(test_images)))

pred_results = trained_model.predict(
    source=[str(p) for p in test_sample],
    imgsz=IMG_SIZE,
    conf=0.25,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(predict_dir),
    name='hit_uav_test_predictions',
    exist_ok=True,
    save=True,
)

print('saved to:', predict_dir / 'hit_uav_test_predictions')

## Preview Predictions

In [ ]:
prediction_images = list((predict_dir / 'hit_uav_test_predictions').glob('*.jpg'))[:6]

plt.figure(figsize=(15, 8))
for idx, image_path in enumerate(prediction_images, start=1):
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, idx)
    plt.imshow(image)
    plt.axis('off')
    plt.title(image_path.name)
plt.tight_layout()
plt.show()

## Export And Package Results

This creates downloadable files under `/kaggle/working/`.

In [ ]:
export_onnx = True

if export_onnx:
    trained_model.export(format='onnx', imgsz=IMG_SIZE)

package_dir = Path('/kaggle/working/package')
package_dir.mkdir(exist_ok=True)

shutil.copy2(best_weights, package_dir / 'hit_uav_yolo11s_best.pt')
if last_weights.exists():
    shutil.copy2(last_weights, package_dir / 'hit_uav_yolo11s_last.pt')
if (best_weights.with_suffix('.onnx')).exists():
    shutil.copy2(best_weights.with_suffix('.onnx'), package_dir / 'hit_uav_yolo11s_best.onnx')
shutil.copy2(DATA_YAML, package_dir / 'hit_uav_thermal.yaml')

shutil.make_archive('/kaggle/working/hit_uav_yolo11s_results', 'zip', package_dir)

print('Package:', '/kaggle/working/hit_uav_yolo11s_results.zip')
print('Best weights:', package_dir / 'hit_uav_yolo11s_best.pt')

## Use The Trained Model In Your Local Demo

Download `hit_uav_yolo11s_best.pt` from Kaggle and place it in your local project, for example:

```text
models/hit_uav_yolo11s_best.pt
```

Then run local video inference:

```bash
./scripts/run_demo.sh models/hit_uav_yolo11s_best.pt data/demo_footage/thermal_demo.mp4 outputs/thermal_demo_annotated.mp4
```